# EDA enriquecido — Dataset Olist (Brazilian E-Commerce)
## Proyecto Ecommify — Diseño y Optimización de Bases de Datos

**Versión:** 2 (enriquecida respecto al EDA de Unidad 1)
**Autor del notebook:** Carlos Arévalo

---

## Sección 0 — Contexto y objetivos del EDA

Este EDA cumple dos funciones:

1. **Caracterizar el dataset Olist** que sirve como base para el diseño de Ecommify (volúmenes, calidad, patrones temporales y geográficos, cardinalidades reales).
2. **Generar evidencia empírica** que justifique las decisiones de diseño tomadas en las Fases 2 a 7 del plan: particionamiento de `orders`, uso de JSONB y arrays en `Product`, separación de `Category` como entidad, resolución de la ambigüedad `customer_id` vs `customer_unique_id`, asignación PostgreSQL vs MongoDB.

Las conclusiones del notebook se consolidan en la **Sección 7**, que sirve de input directo al `01_analisis_requisitos.md` y al `02_descripcion_entidades.md`.

**Diferencias respecto al EDA de Unidad 1:** el EDA anterior cubrió solo distribución temporal mensual, distribución geográfica por estado y distribución de precios. Esta versión incorpora análisis de calidad de datos por columna con porcentajes, detección de outliers vía IQR, cardinalidades reales entre tablas, patrones de día de semana, cruces de variables (review vs entrega, freight vs estado) y conclusiones explícitamente vinculadas al diseño.

---
## Sección 1 — Carga e inventario de datos

Las 9 tablas del dataset y su rol en el dominio:

| Tabla | Rol en el dominio | PK candidata |
|---|---|---|
| `customers` | Identidad y ubicación de cliente (por orden) | `customer_id` |
| `orders` | Cabecera de orden con ciclo de vida | `order_id` |
| `order_items` | Líneas de detalle por orden | (`order_id`, `order_item_id`) |
| `order_payments` | Pagos por orden, soporta múltiples métodos | (`order_id`, `payment_sequential`) |
| `order_reviews` | Reseñas de cliente | `review_id` |
| `products` | Catálogo de productos | `product_id` |
| `sellers` | Vendedores | `seller_id` |
| `geolocation` | Mapeo zip → lat/lng | (`zip_code_prefix`, lat, lng) |
| `product_category_name_translation` | Traducción de categorías | `product_category_name` |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

BASE_PATH = '/content/drive/MyDrive/master/Diseno y optimizacion de DB'
print('Archivos disponibles:')
for f in sorted(os.listdir(BASE_PATH)):
    print(' -', f)

In [ ]:
# Carga de las 9 tablas
customers = pd.read_csv(f'{BASE_PATH}/olist_customers_dataset.csv')
geolocation = pd.read_csv(f'{BASE_PATH}/olist_geolocation_dataset.csv')
order_items = pd.read_csv(f'{BASE_PATH}/olist_order_items_dataset.csv')
order_payments = pd.read_csv(f'{BASE_PATH}/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(f'{BASE_PATH}/olist_order_reviews_dataset.csv')
orders = pd.read_csv(f'{BASE_PATH}/olist_orders_dataset.csv')
products = pd.read_csv(f'{BASE_PATH}/olist_products_dataset.csv')
sellers = pd.read_csv(f'{BASE_PATH}/olist_sellers_dataset.csv')
category_translation = pd.read_csv(f'{BASE_PATH}/product_category_name_translation.csv')

# Conversión de timestamps - crítico para análisis temporal
date_cols_orders = ['order_purchase_timestamp', 'order_approved_at',
                    'order_delivered_carrier_date', 'order_delivered_customer_date',
                    'order_estimated_delivery_date']
for c in date_cols_orders:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'], errors='coerce')
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'], errors='coerce')

tables = {
    'customers': customers, 'geolocation': geolocation, 'order_items': order_items,
    'order_payments': order_payments, 'order_reviews': order_reviews,
    'orders': orders, 'products': products, 'sellers': sellers,
    'category_translation': category_translation
}

print('Inventario:')
for name, df in tables.items():
    print(f'  {name:30s} {df.shape[0]:>10,} filas  {df.shape[1]:>3} columnas  {df.memory_usage(deep=True).sum() / 1024**2:>7.1f} MB')

---
## Sección 2 — Calidad de datos

Para cada tabla revisamos:
- Porcentaje de nulos por columna.
- Duplicados sobre PKs candidatas.
- Tipos correctos vs inferidos.
- Valores fuera de rango lógico (negativos en montos, fechas futuras, scores fuera de [1,5]).

In [ ]:
def quality_report(name, df, pk=None):
    print(f'\n=== {name} ===')
    print(f'Filas: {len(df):,} | Columnas: {len(df.columns)}')
    nulls = (df.isnull().sum() / len(df) * 100).round(2)
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if len(nulls):
        print('Nulos (%):')
        print(nulls.to_string())
    else:
        print('Nulos: ninguno.')
    if pk:
        dup = df.duplicated(subset=pk).sum()
        unique = df.drop_duplicates(subset=pk).shape[0]
        print(f'PK candidata {pk}: {unique:,} únicas | duplicadas: {dup}')

quality_report('customers', customers, pk=['customer_id'])
quality_report('orders', orders, pk=['order_id'])
quality_report('order_items', order_items, pk=['order_id', 'order_item_id'])
quality_report('order_payments', order_payments, pk=['order_id', 'payment_sequential'])
quality_report('order_reviews', order_reviews, pk=['review_id'])
quality_report('products', products, pk=['product_id'])
quality_report('sellers', sellers, pk=['seller_id'])

In [ ]:
# Hallazgo clave: customer_id vs customer_unique_id
n_cust_id = customers['customer_id'].nunique()
n_unique_id = customers['customer_unique_id'].nunique()
ratio = n_cust_id / n_unique_id
print(f'customer_id distintos:        {n_cust_id:,}')
print(f'customer_unique_id distintos: {n_unique_id:,}')
print(f'Ratio: {ratio:.3f} órdenes por cliente real')
print()
print('Interpretación: customer_id es identificador POR ORDEN, no por cliente real.')
print('customer_unique_id es el cliente real. Esto justifica que en el ER de U2')
print('Customer tenga PK customer_unique_id, no customer_id.')

In [ ]:
# Valores fuera de rango lógico
print('Precios negativos en order_items:', (order_items['price'] < 0).sum())
print('Freight negativo en order_items:', (order_items['freight_value'] < 0).sum())
print('Payment value negativo:', (order_payments['payment_value'] < 0).sum())
print('Review scores fuera de [1,5]:', (~order_reviews['review_score'].between(1, 5)).sum())
print('Fechas de entrega antes de compra:',
      ((orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days < 0).sum())

---
## Sección 3 — Análisis univariado

Distribuciones y outliers de las variables clave.

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = ((s < lower) | (s > upper)).sum()
    return {'q1': q1, 'median': s.median(), 'q3': q3, 'iqr': iqr,
            'lower_bound': lower, 'upper_bound': upper,
            'outliers': out, 'outliers_pct': out / len(s) * 100}

for var, s in [('price', order_items['price']),
               ('freight_value', order_items['freight_value']),
               ('payment_value', order_payments['payment_value']),
               ('payment_installments', order_payments['payment_installments'])]:
    r = iqr_outliers(s.dropna())
    print(f'\n{var}:')
    for k, v in r.items():
        print(f'  {k:18s} {v:>12,.2f}' if isinstance(v, float) else f'  {k:18s} {v}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].hist(order_items['price'].dropna(), bins=80, edgecolor='black')
axes[0,0].set_yscale('log')
axes[0,0].set_title('Distribución de price (escala log en y)')
axes[0,0].set_xlabel('Precio (BRL)')

axes[0,1].hist(order_items['freight_value'].dropna(), bins=60, edgecolor='black', color='orange')
axes[0,1].set_title('Distribución de freight_value')
axes[0,1].set_xlabel('Valor de flete (BRL)')

review_dist = order_reviews['review_score'].value_counts().sort_index()
axes[1,0].bar(review_dist.index.astype(str), review_dist.values, color='steelblue', edgecolor='black')
axes[1,0].set_title('Distribución de review_score')
axes[1,0].set_xlabel('Score')
axes[1,0].set_ylabel('Frecuencia')

axes[1,1].hist(order_payments['payment_installments'].dropna(), bins=24, edgecolor='black', color='seagreen')
axes[1,1].set_title('Distribución de payment_installments')
axes[1,1].set_xlabel('Cuotas')

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de payment_type
pt = order_payments['payment_type'].value_counts(normalize=True) * 100
print('Distribución de payment_type (%):')
print(pt.round(2).to_string())
print()
print('Decisión de diseño: este conjunto cerrado de valores justifica modelar')
print('payment_type como ENUM en PostgreSQL (credit_card, debit_card, boleto, voucher, pix).')

---
## Sección 4 — Análisis bivariado y de relaciones

Cardinalidades reales entre tablas (fundamental para fijar relaciones del ER) y cruces de variables que informan decisiones arquitectónicas.

In [ ]:
# Items por orden
items_per_order = order_items.groupby('order_id').size()
print('Items por orden:')
print(items_per_order.describe().round(2).to_string())
print(f'\nDistribución:')
print(items_per_order.value_counts().head(10).sort_index().to_string())
print(f'\nÓrdenes con >5 items: {(items_per_order > 5).sum():,}')
print(f'Órdenes con >10 items: {(items_per_order > 10).sum():,}')

In [ ]:
# Órdenes por cliente real (customer_unique_id)
orders_cust = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
orders_per_cust = orders_cust.groupby('customer_unique_id').size()
print('Órdenes por cliente real:')
print(orders_per_cust.describe().round(2).to_string())
print(f'\nClientes con 1 orden: {(orders_per_cust == 1).sum():,} ({(orders_per_cust == 1).mean()*100:.1f}%)')
print(f'Clientes con 2+ órdenes: {(orders_per_cust >= 2).sum():,} ({(orders_per_cust >= 2).mean()*100:.1f}%)')
print(f'Clientes con 5+ órdenes: {(orders_per_cust >= 5).sum():,}')

In [ ]:
# Reviews por orden
reviews_per_order = order_reviews.groupby('order_id').size()
print('Reviews por orden:')
print(reviews_per_order.describe().round(2).to_string())
print(f'\nÓrdenes con review: {reviews_per_order.shape[0]:,} de {orders.shape[0]:,} ({reviews_per_order.shape[0]/orders.shape[0]*100:.1f}%)')
print(f'Órdenes con múltiples reviews: {(reviews_per_order > 1).sum():,}')
print('\nDecisión de diseño: relación 1:N entre Order y Review (no 1:1 como podría asumirse).')

In [ ]:
# Pagos por orden
payments_per_order = order_payments.groupby('order_id').size()
print('Pagos por orden:')
print(payments_per_order.describe().round(2).to_string())
print(f'\nÓrdenes con 1 método de pago: {(payments_per_order == 1).sum():,} ({(payments_per_order == 1).mean()*100:.1f}%)')
print(f'Órdenes con múltiples métodos: {(payments_per_order > 1).sum():,}')

In [ ]:
# Categorías de producto: caso para tabla Category separada
n_cats = products['product_category_name'].nunique(dropna=True)
print(f'Categorías distintas en products: {n_cats}')
print(f'Productos sin categoría: {products["product_category_name"].isna().sum():,}')
print()
top_cats = products['product_category_name'].value_counts().head(10)
print('Top 10 categorías por # productos:')
print(top_cats.to_string())
print()
print('Decisión de diseño: 71 categorías repetidas en 32k productos =')
print('alta redundancia. Justifica extraer Category como tabla aparte (3FN).')

In [ ]:
# Review score vs tiempo de entrega
od = orders[orders['order_delivered_customer_date'].notna()].copy()
od['delivery_days'] = (od['order_delivered_customer_date'] - od['order_purchase_timestamp']).dt.days
od_rev = od.merge(order_reviews[['order_id', 'review_score']], on='order_id')

review_vs_delivery = od_rev.groupby('review_score')['delivery_days'].agg(['mean', 'median', 'count']).round(2)
print('Review score vs días de entrega:')
print(review_vs_delivery.to_string())
print()
print('Hallazgo: existe relación inversa entre tiempo de entrega y score.')
print('Esto justifica capturar timestamps de entrega como atributos críticos en Order.')

---
## Sección 5 — Patrones temporales

Volumen de órdenes por mes, día de semana, hora del día. Sustenta el particionamiento por fecha de la tabla `orders`.

In [ ]:
orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
orders['weekday'] = orders['order_purchase_timestamp'].dt.day_name()
orders['hour'] = orders['order_purchase_timestamp'].dt.hour

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

monthly = orders['year_month'].value_counts().sort_index()
axes[0].bar(monthly.index.astype(str), monthly.values, edgecolor='black')
axes[0].set_title('Órdenes por mes')
axes[0].tick_params(axis='x', rotation=70)

wd_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday = orders['weekday'].value_counts().reindex(wd_order)
axes[1].bar(weekday.index, weekday.values, color='orange', edgecolor='black')
axes[1].set_title('Órdenes por día de semana')

hourly = orders['hour'].value_counts().sort_index()
axes[2].bar(hourly.index, hourly.values, color='seagreen', edgecolor='black')
axes[2].set_title('Órdenes por hora del día')
axes[2].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

In [ ]:
# Volumen acumulado: justifica particiones mensuales sobre orders
monthly_cum = monthly.cumsum()
print('Volumen acumulado de órdenes por mes:')
print(monthly_cum.tail(10).to_string())
print(f'\nTotal órdenes: {monthly.sum():,}')
print(f'Periodo cubierto: {orders["order_purchase_timestamp"].min()} a {orders["order_purchase_timestamp"].max()}')
print()
print('Decisión: particionar orders por RANGE(order_purchase_timestamp) con particiones')
print('mensuales para los últimos 12 meses (hot) y anuales para histórico (cold).')

---
## Sección 6 — Patrones geográficos

Distribución de clientes y sellers. Sustenta el uso de PostGIS para cálculo de distancia vendedor-cliente.

In [ ]:
cust_state = customers['customer_state'].value_counts()
sel_state = sellers['seller_state'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(cust_state.head(15).index, cust_state.head(15).values, edgecolor='black')
axes[0].set_title('Top 15 estados por clientes')

axes[1].bar(sel_state.head(15).index, sel_state.head(15).values, color='orange', edgecolor='black')
axes[1].set_title('Top 15 estados por sellers')

plt.tight_layout()
plt.show()

print(f'\nClientes en SP: {cust_state["SP"]:,} ({cust_state["SP"]/cust_state.sum()*100:.1f}%)')
print(f'Sellers en SP:  {sel_state["SP"]:,} ({sel_state["SP"]/sel_state.sum()*100:.1f}%)')

In [ ]:
# Matriz origen-destino: seller_state -> customer_state
flow = order_items.merge(orders[['order_id', 'customer_id']], on='order_id')\
                  .merge(customers[['customer_id', 'customer_state']], on='customer_id')\
                  .merge(sellers[['seller_id', 'seller_state']], on='seller_id')

matrix = flow.groupby(['seller_state', 'customer_state']).size().unstack(fill_value=0)
top_seller_states = matrix.sum(axis=1).nlargest(8).index
top_cust_states = matrix.sum(axis=0).nlargest(8).index
sub = matrix.loc[top_seller_states, top_cust_states]

plt.figure(figsize=(10, 6))
sns.heatmap(sub, annot=True, fmt=',', cmap='Blues', cbar_kws={'label': 'Items vendidos'})
plt.title('Flujo seller_state → customer_state (top 8 x top 8)')
plt.xlabel('Estado del cliente')
plt.ylabel('Estado del seller')
plt.tight_layout()
plt.show()

print('\nDecisión: el flujo geográfico no es uniforme, hay distancias largas frecuentes.')
print('Justifica usar PostGIS para cálculo eficiente de distancia y optimización de costo de envío.')

---
## Sección 7 — Conclusiones que informan el diseño

Esta sección consolida los hallazgos del EDA en decisiones de diseño concretas. Es input directo de las Fases 2 a 7 del plan.

### 7.1 Volúmenes esperados

| Entidad | Volumen Olist | Crecimiento esperado Ecommify | Decisión de diseño |
|---|---|---|---|
| Customer (real) | ~96k unique_ids | +30%/año | Tabla normal en PostgreSQL |
| Order | ~100k | +50%/año | Particionar por `order_purchase_timestamp` |
| OrderItem | ~113k | +50%/año | Conjunta con Order, no particionar |
| Product | ~33k | +20%/año | Tipos avanzados (JSONB, ARRAY) en PostgreSQL |
| Review | ~100k | +50%/año | MongoDB (alta escritura, contenido variable) |
| Seller | ~3k | +10%/año | Tabla normal en PostgreSQL |
| Payment | ~104k | +50%/año | PostgreSQL ACID estricto |

### 7.2 Hallazgos clave y su impacto en el diseño

1. **`customer_id` vs `customer_unique_id`.** El ratio de ~1.03 órdenes por cliente confirma que `customer_id` es identificador por orden (sesión de compra), no por cliente real. En el ER de U2, la entidad `Customer` tiene PK `customer_unique_id`. El `customer_id` de Olist se trata como atributo histórico de sesión o se descarta.

2. **Categorías repetidas masivamente.** 71 categorías distintas cubriendo 32k productos = redundancia alta. En U1 quedó como atributo dentro de `Product` (violación 3FN). En U2 se corrige extrayendo `Category` como tabla aparte con PK propia.

3. **Reviews 1:N con Order.** Existen órdenes con múltiples reviews. El ER refleja relación 1:N, no 1:1.

4. **Pagos 1:N con Order.** ~99% de órdenes tienen 1 pago, pero hay casos con múltiples métodos. Tabla `Payment` con PK compuesta (`order_id`, `payment_sequential`).

5. **`payment_type` con dominio cerrado.** Justifica modelarlo como ENUM en PostgreSQL.

6. **Volúmenes y patrón temporal de `orders`.** Crecimiento mensual sostenido + ventana histórica de ~2 años de datos = patrón típico de hot/cold partitioning. Particionar por mes en hot, por año en cold.

7. **Atributos físicos de producto variables por categoría.** Pesos y dimensiones varían fuertemente. Esto, sumado al hecho de que productos electrónicos vs ropa vs hogar tienen atributos no compartidos, justifica `product_specifications JSONB` como decidió Andrés Camilo en la Formativa.

8. **Flujos geográficos no uniformes.** SP concentra >40% de clientes y sellers, pero hay tráfico relevante entre todos los estados. PostGIS permite calcular distancia real para optimizar `freight_value`, en vez de aproximar por estado.

9. **Score de review correlacionado con tiempo de entrega.** Justifica capturar todos los timestamps del ciclo de vida de Order como atributos críticos.

10. **Outliers de precio y flete.** Hay valores extremos pero legítimos (productos premium, envíos cross-country). No se filtran, se aceptan en el modelo. CHECK constraints solo evitan negativos.

### 7.3 Asignación PostgreSQL vs MongoDB derivada del EDA

- **PostgreSQL (fuente de verdad):** Customer, Order (particionada), OrderItem, Product, Seller, Payment, Category, Promotion. Todos con tipos avanzados nativos donde aplique.
- **MongoDB (complementario):** proyección de catálogo para frontend, Reviews, analytics_events, user_sessions.